<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week0_setup_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 0 — Setup Check
### The NLP Internship | LinguaAI Labs

Run every cell. All three tests must pass before Week 1.

In [ ]:
# ── PLATFORM SETUP ────────────────────────────────────────────────────────────
# Run this cell first. Installs missing libraries in Google Colab.
# In a local environment with requirements.txt installed, you can skip this.
import sys, subprocess, importlib

def install_if_missing(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

install_if_missing("spacy")
install_if_missing("nltk")
install_if_missing("langdetect")
install_if_missing("transformers")
install_if_missing("datasets")
install_if_missing("torch")
install_if_missing("gensim")
install_if_missing("pyldavis", "pyLDAvis")
install_if_missing("fastapi")
install_if_missing("uvicorn")

import nltk, os
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger"]:
    nltk.download(resource, quiet=True)
print("Environment ready ✅")

## Test 1 — spaCy

In [ ]:
import spacy

# Download the English model if not present
import subprocess, sys
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
    nlp = spacy.load("en_core_web_sm")

doc = nlp("Customer says the app keeps crashing every time she tries to pay.")

print("Tokens and POS tags:")
for token in doc:
    print(f"  {token.text:<20} POS: {token.pos_:<10} Lemma: {token.lemma_}")
print("\nTest 1 PASSED ✅")

## Test 2 — Hugging Face Pipeline

In [ ]:
from transformers import pipeline

# Pre-trained sentiment model (downloads ~260MB first time)
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

tickets = [
    "I cannot believe how terrible this service is. Three days and no response.",
    "Thank you for sorting this so quickly! Really appreciated.",
    "My account is locked and I need access urgently please",
]

print("Sentiment predictions (pre-trained on movie reviews — note domain mismatch):")
for text, result in zip(tickets, classifier(tickets)):
    print(f"  [{result['label']:8} {result['score']:.2f}] {text[:60]}")

print("\nTest 2 PASSED ✅")
print("Note: a complaint about a crashing app may be predicted POSITIVE —")
print("that gap between pre-trained and task-specific is exactly what this book closes.")

## Test 3 — Dataset Loads

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/support_tickets.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/nlp-internship-in-a-book/main/data/support_tickets.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload support_tickets.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        tickets = [
            'My account login is not working, please help',
            'I was charged twice for my subscription this month',
            'The app crashes whenever I try to upload a file',
            'How do I cancel my subscription?',
            'Great service, very happy with the product!',
            'I cannot reset my password, the email never arrives',
            'The dashboard is loading very slowly today',
            'I need a refund for my last order',
            'Feature request: can you add dark mode?',
            'Error 500 when trying to export my data',
        ] * 50
        np.random.shuffle(tickets)
        df = pd.DataFrame({
            'ticket_id':  range(1, 501),
            'text':       tickets[:500],
            'category':   np.random.choice(['billing','technical','account','general'], 500),
            'sentiment':  np.random.choice(['positive','negative','neutral'], 500, p=[0.3,0.4,0.3]),
            'priority':   np.random.choice(['low','medium','high'], 500),
        })
        print('Sample dataset ready (500 tickets) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


## Tokenisation Deep Dive

> 🔬 **Three tokenisation levels** — understand these before Week 1 begins.

In [ ]:
import spacy
from transformers import AutoTokenizer

nlp = spacy.load("en_core_web_sm")
bert_tok  = AutoTokenizer.from_pretrained("bert-base-uncased")
mbert_tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

test_sentences = [
    ("English",  "my account has been locked for three days"),
    ("Pidgin",   "Abeg una support line no dey pick"),
    ("Yoruba-mix","E joo help me mo ti pay but e no go through"),
]

print(f"{'Text type':<12} {'spaCy':>8} {'BERT':>8} {'mBERT':>8}")
print("-" * 42)
for label, sent in test_sentences:
    sp  = len([t for t in nlp(sent) if not t.is_space])
    bt  = len(bert_tok.tokenize(sent))
    mbt = len(mbert_tok.tokenize(sent))
    print(f"  {label:<10} {sp:>8} {bt:>8} {mbt:>8}")

print("\nFewer mBERT tokens = better vocabulary coverage for that language.")

## ✅ What You Can Do After This Week

- Install and confirm the NLP environment
- Tokenise text at word, subword, and character level
- Explain why a pre-trained model fails on a new domain
- Load the CustomerCare-50K corpus

---
## ✅ Week 0 Complete
**Next:** `week1_text_is_not_data_STARTER.ipynb`

---
*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

*Add this week's notebook to your internship portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
